# Mapping each sample on the atlas

In [3]:
# Load packages
suppressPackageStartupMessages({
    library(dplyr)
    library(data.table)
    library(ggplot2)
    library(SingleCellExperiment)
    library(scater)
    library(scran)
    library(edgeR)
    library(ggrastr)
    library(batchelor)
})

here::i_am("mapping/run/01_mapping_manual.ipynb")

# Load mapping functions
source(here::here("mapping/run/mnn/mapping_functions_extended.R"))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

set.seed(1234)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/12_Eomes_T_Mixl1/T_E75/code



In [4]:
args = list()
args$atlas_sce = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/processed/SingleCellExperiment.rds'
args$atlas_metadata = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/sample_metadata_extended.txt.gz'
args$atlas_stages = c('E6.5','E6.75','E7.0','E7.25', 'E7.5', 'E7.75', 'E8.0', 'E8.25', 'E8.5')
args$npcs = 40
args$n_neighbours = 15
args$cosine_normalisation = FALSE

args$sce = io$rna.sce
args$metadata =  paste0(io$basedir,"results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz")
args$outdir = paste0(io$basedir,"/results/rna/mapping/manual/")
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [5]:
################
## Load query ##
################

# Get query data
meta_query = fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_score <= 0.5] # Resetting doublet score threshold
# Load RNA expression data as SingleCellExperiment object
sce_query <- load_SingleCellExperiment(args$sce, cells=meta_query$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce_query) <- meta_query %>% tibble::column_to_rownames("cell") %>% DataFrame

In [6]:
################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)] 

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

In [7]:
#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

rownames(sce_atlas) = gene_metadata[match(rownames(sce_atlas), ens_id), symbol]

In [8]:
# Imprinted genes
# imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
#                        grep('paternally', gene_metadata$description)), symbol]
#Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)] # filter out non-informative genes
genes.intersect <- genes.intersect[grep("^Hbb|^Hba",genes.intersect,invert=T)] # test removing Haem genes 
genes.intersect <- genes.intersect[!genes.intersect %in% c('Grb10', 'Nnat')] # remove imprinted genes
genes.intersect <- genes.intersect[!genes.intersect %in% c("Xist", "Tsix")] # remove Xist & Tsix
genes.intersect <- genes.intersect[!genes.intersect=="tomato-td"] # remove tomato itself
genes.intersect <- genes.intersect[!genes.intersect %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [9]:
#######################
## Feature selection ##
#######################

# Load gene statistics from the atlas
gene_stats.dt <- fread(paste0(io$atlas.basedir,"/results/gene_statistics/gene_statistics.txt.gz")) %>%
.[gene%in%genes.intersect]
genes_to_use <- gene_stats.dt %>% setorder(-var_pseudobulk, na.last = T) %>% head(n=4000) %>% .$gene  

stopifnot(genes_to_use%in%rownames(sce_atlas))
stopifnot(genes_to_use%in%rownames(sce_query))

sce_query = sce_query[genes_to_use,]
sce_atlas = sce_atlas[genes_to_use,]

In [10]:
sce_query
sce_atlas

class: SingleCellExperiment 
dim: 4000 4139 
metadata(0):
assays(2): counts logcounts
rownames(4000): Tdgf1 Hoxaas3 ... Ppp2r3a Upk1b
rowData names(0):
colnames(4139): sample_11#AAACGGGTCCCTTGTG sample_11#AAAGCAAGTACGACCC
  ... sample_16#TTTGTCAAGACTTGAA sample_16#TTTGTCAGTAGCCTAT
colData names(13): sample barcode ... doublet_score doublet_call
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

class: SingleCellExperiment 
dim: 4000 108857 
metadata(1): log.exprs.offset
assays(2): counts logcounts
rownames(4000): Tdgf1 Hoxaas3 ... Ppp2r3a Upk1b
rowData names(0):
colnames(108857): cell_1 cell_2 ... cell_139330 cell_139331
colData names(13): barcode sample ... index celltype_extended
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [11]:
mapping_full = mclapply(unique(meta_query$sample), function(x){
    mapping  <- mapWrap(
          sce_atlas = sce_atlas,
          meta_atlas = meta_atlas,
          sce_query = sce_query[,meta_query[sample==x, cell]],
          meta_query = meta_query[sample==x, ],
          genes = genes_to_use,
          npcs = args$npcs,
          k = args$n_neighbours,
          cosineNorm = args$cosine_normalisation,
          order = NULL
        )
    
    mapping.dt <- mapping$mapping %>% 
      .[,c("cell","celltype.mapped","celltype.score","celltype_extended.mapped","celltype_extended.score","stage.mapped", "cellstage.score", "closest.cell")] %>% 
      as.data.table
    
    return(mapping.dt)
}, mc.cores=4)

In [12]:
mapping_full = mapping_full %>% rbindlist()

In [13]:
mapping_full = mapping_full %>% setnames(colnames(mapping_full)[-1], paste0(colnames(mapping_full)[-1], '_mnn'))

In [14]:
meta_full = merge(meta_query, mapping_full, by='cell') %>% 
    .[match(colnames(sce_query), cell)]

In [16]:
fwrite(meta_full, file.path(args$outdir, 'sample_metadata_after_mapping.txt.gz'), sep="\t")

In [15]:
args$outdir

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/12_Eomes_T_Mixl1/T_E75//results/rna/mapping/manual/"